<a href="https://colab.research.google.com/github/olajidechris/CollabProjects/blob/main/Project_A_Search_friction_pipeline_(latest).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### A Pipeline for identifying "Zero-Result" Frictions & Search Abandonment

Merchandisers lose massive amounts of money when users search for a product, get zero results (or irrelevant ones), and bounce. This project proves you can track user friction through an e-commerce funnel at scale.

* **The Business Value:** Providing merchandisers with a dashboard identifying which search terms or product categories have the highest cart-abandonment rates, allowing them to manually adjust search synonyms or algorithms to recover lost revenue.
* **The Dataset:** Kaggle’s *“eCommerce behavior data from multi category store”* (contains hundreds of millions of rows of user event data: view, cart, purchase).
* **Methods:**
* Use **PySpark Window Functions** to perform sessionization, grouping chronological user events (e.g., view -> view -> cart -> abandon) into distinct shopping sessions.
* Calculate the exact conversion drop-off rates at each stage of the funnel.
* Use PySpark SQL to aggregate lost revenue by specific product categories.

* **The Deliverable:** A simulated dashboard report (using Matplotlib/Seaborn or a linked Looker Studio/Tableau public link) highlighting the top 5 product categories bleeding revenue due to interaction friction, complete with unit economic projections of what a 5% recovery would yield.


### Setting up modules and importing datasets


In [ ]:
try:
  import kagglehub
  # kagglehub.dataset_download('<owner>/<dataset-slug>')
  dataset_path = kagglehub.dataset_download('mkechinov/ecommerce-behavior-data-from-multi-category-store')
  print("Path to dataset files:", dataset_path)
except ModuleNotFoundError:
  print("Kagglehub module imports failed.")

100%|██████████| 4.29G/4.29G [03:19<00:00, 23.1MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store/versions/8


In [ ]:
import os
file_paths = []
for dirname, _, filenames in os.walk(dataset_path):
    for filename in sorted(filenames):
        file_path = os.path.join(dirname, filename)
        file_paths += [file_path]
        print(file_path)

/root/.cache/kagglehub/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store/versions/8/2019-Nov.csv
/root/.cache/kagglehub/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store/versions/8/2019-Oct.csv


In [ ]:
try:
  import pyspark.sql.functions as sf
  import pyspark.sql.types as st
  from pyspark.sql import SparkSession
  from pyspark.sql.window import Window
  print("Pyspark modules imported successfully.")
except ModuleNotFoundError:
  print("Pyspark module imports failed.")

Pyspark modules imported successfully.


#### Loading and examining datasets

In [ ]:
spark = SparkSession.builder \
  .appName("SearchFriction") \
  .master("local[*]") \
  .config("spark.sparkContext.setLogLevel", "WARN") \
  .getOrCreate()

spark

In [ ]:
# load data for Nov 2019
df1 = spark.read.csv(file_paths[0], header=True, inferSchema=True)
df1.show(5)
df1.printSchema()


+-------------------+----------+----------+-------------------+--------------------+------+------+---------+--------------------+
|         event_time|event_type|product_id|        category_id|       category_code| brand| price|  user_id|        user_session|
+-------------------+----------+----------+-------------------+--------------------+------+------+---------+--------------------+
|2019-11-01 00:00:00|      view|   1003461|2053013555631882655|electronics.smart...|xiaomi|489.07|520088904|4d3b30da-a5e4-49d...|
|2019-11-01 00:00:00|      view|   5000088|2053013566100866035|appliances.sewing...|janome|293.65|530496790|8e5f4f83-366c-4f7...|
|2019-11-01 00:00:01|      view|  17302664|2053013553853497655|                NULL| creed| 28.31|561587266|755422e7-9040-477...|
|2019-11-01 00:00:01|      view|   3601530|2053013563810775923|appliances.kitche...|    lg|712.87|518085591|3bfb58cd-7892-48c...|
|2019-11-01 00:00:01|      view|   1004775|2053013555631882655|electronics.smart...|xiaomi

In [ ]:
# load data for Oct 2019
df2 = spark.read.csv(file_paths[1], header=True, inferSchema=True)
df2.show(5)
df2.printSchema()


+-------------------+----------+----------+-------------------+--------------------+--------+-------+---------+--------------------+
|         event_time|event_type|product_id|        category_id|       category_code|   brand|  price|  user_id|        user_session|
+-------------------+----------+----------+-------------------+--------------------+--------+-------+---------+--------------------+
|2019-10-01 00:00:00|      view|  44600062|2103807459595387724|                NULL|shiseido|  35.79|541312140|72d76fde-8bb3-4e0...|
|2019-10-01 00:00:00|      view|   3900821|2053013552326770905|appliances.enviro...|    aqua|   33.2|554748717|9333dfbd-b87a-470...|
|2019-10-01 00:00:01|      view|  17200506|2053013559792632471|furniture.living_...|    NULL|  543.1|519107250|566511c2-e2e3-422...|
|2019-10-01 00:00:01|      view|   1307067|2053013558920217191|  computers.notebook|  lenovo| 251.74|550050854|7c90fc70-0e80-459...|
|2019-10-01 00:00:04|      view|   1004237|2053013555631882655|electr

In [ ]:
print(f'October dataset is: {df1.count()} rows. \n',
      f'October dataset is: {df2.count()} rows. \n')

October dataset is: 67501979 rows. 
 October dataset is: 42448764 rows. 



In [ ]:
# Combine datasets and del separate instances
df = df1.union(df2)
del df1
del df2

# save combined dataset as parquet to speed up processing
parquet_path = '/kaggle/working/dataset.parquet'
df.write.parquet("/kaggle/working/dataset.parquet")

In [ ]:
# reload dataset
df = spark.read.parquet(parquet_path)

#df.cache()

df.show(5)
df.printSchema()
df.count()

+-------------------+----------+----------+-------------------+--------------------+--------+-------+---------+--------------------+
|         event_time|event_type|product_id|        category_id|       category_code|   brand|  price|  user_id|        user_session|
+-------------------+----------+----------+-------------------+--------------------+--------+-------+---------+--------------------+
|2019-11-17 08:43:00|      view|   2501799|2053013564003713919|appliances.kitche...|elenberg|  46.31|563237118|4368d099-6d19-47c...|
|2019-11-17 08:43:00|      view|   6400335|2053013554121933129|computers.compone...|   intel| 435.28|551129779|4db2c365-ee85-443...|
|2019-11-17 08:43:00|      view|   3701538|2053013565983425517|appliances.enviro...|  irobot|1878.81|539845715|bf7d95c0-69e1-40f...|
|2019-11-17 08:43:00|      view|  26400266|2053013563651392361|                NULL| lucente| 119.18|572211322|8e6c63f8-7f34-48b...|
|2019-11-17 08:43:00|      view|   1004659|2053013555631882655|electr

109950743

In [ ]:
# Define the window
window_spec = Window.partitionBy("user_session").orderBy("event_time")

### Clean data

In [ ]:
# Data Cleaning: Check for nulls and filter out rows without critical session info

print(f"Total records before cleaning: {df.count()}")

# Check for nulls in columns
null_counts = df.select([sf.count(sf.when(sf.col(c).isNull(), c)).alias(c) for c in df.columns])
null_counts.show()

# Filter out rows where user_session or user_id is null
df_cleaned = df.filter(sf.col("user_session").isNotNull() & sf.col("user_id").isNotNull())

print(f"Total records after cleaning: {df_cleaned.count()}")

# Check for nulls in other columns for awareness
null_counts = df_cleaned.select([sf.count(sf.when(sf.col(c).isNull(), c)).alias(c) for c in df_cleaned.columns])
null_counts.show()


Total records before cleaning: 109950743
+----------+----------+----------+-----------+-------------+--------+-----+-------+------------+
|event_time|event_type|product_id|category_id|category_code|   brand|price|user_id|user_session|
+----------+----------+----------+-----------+-------------+--------+-----+-------+------------+
|         0|         0|         0|          0|     35413780|15331243|    0|      0|          12|
+----------+----------+----------+-----------+-------------+--------+-----+-------+------------+

Total records after cleaning: 109950731
+----------+----------+----------+-----------+-------------+--------+-----+-------+------------+
|event_time|event_type|product_id|category_id|category_code|   brand|price|user_id|user_session|
+----------+----------+----------+-----------+-------------+--------+-----+-------+------------+
|         0|         0|         0|          0|     35413777|15331241|    0|      0|           0|
+----------+----------+----------+-----------

In [ ]:
# Preview the distribution of event types
df_cleaned.groupBy("event_type").count().orderBy("count", ascending=False).show()

+----------+---------+
|event_type|    count|
+----------+---------+
|      view|104335509|
|      cart|  3955434|
|  purchase|  1659788|
+----------+---------+



### 1. Advanced Sessionization & Funnel Flags
We will identify for each event if it eventually led to a 'cart' or 'purchase' within the same session.

In [ ]:
# Sessionization: Create lead/lag features to see the user journey

# Adding previous and next event info to each row within the session window
df_sessionized = df_cleaned.withColumn("prev_event", sf.lag("event_type").over(window_spec)) \
                           .withColumn("next_event", sf.lead("event_type").over(window_spec)) \
                           .withColumn("prev_product", sf.lag("product_id").over(window_spec))


In [ ]:
# Identify sessions that ended in a bounce (only one view and no further action)
session_counts = df_cleaned.groupBy("user_session").count()

session_counts.orderBy(sf.desc("count")).show()


+--------------------+-----+
|        user_session|count|
+--------------------+-----+
|d99d91bf-40f8-4e2...| 4128|
|fc749a4e-c432-4da...| 2466|
|b556f0c7-3a23-44f...| 1963|
|d6433d7b-3846-456...| 1658|
|88206fc3-b5ea-4e3...| 1373|
|af4ad1c0-a131-4fa...| 1294|
|fb075266-182d-4c1...| 1159|
|cfb90a35-9575-495...| 1137|
|0c307610-aa79-bf1...|  918|
|8c322efd-e368-42a...|  711|
|58c242f5-c85d-446...|  679|
|53c412c9-e864-d0d...|  594|
|2183f046-46f1-4ff...|  584|
|b2101293-44c1-481...|  564|
|85890756-b4a0-4cc...|  550|
|3a1a8cae-b7bd-444...|  534|
|4488e77a-9901-4c4...|  504|
|965efde8-197f-478...|  485|
|727dbaa7-fa0e-4c2...|  472|
|88bfe8fa-2531-4b0...|  464|
+--------------------+-----+
only showing top 20 rows


In [ ]:
# Create flags for funnel stages
df_funnel = df_cleaned.withColumn("is_view", sf.when(sf.col("event_type") == "view", 1).otherwise(0)) \
    .withColumn("is_cart", sf.when(sf.col("event_type") == "cart", 1).otherwise(0)) \
    .withColumn("is_purchase", sf.when(sf.col("event_type") == "purchase", 1).otherwise(0))

# Aggregate per session to find conversion
session_funnel = df_funnel.groupBy("user_session", "category_code") \
    .agg(sf.max("is_view").alias("has_view"),
         sf.max("is_cart").alias("has_cart"),
         sf.max("is_purchase").alias("has_purchase"),
         sf.sum(sf.when(sf.col("event_type") == "view", sf.col("price")).otherwise(0)).alias("potential_revenue"))

session_funnel.show(5)

+--------------------+--------------------+--------+--------+------------+------------------+
|        user_session|       category_code|has_view|has_cart|has_purchase| potential_revenue|
+--------------------+--------------------+--------+--------+------------+------------------+
|a0f5e1bb-bf99-475...|electronics.smart...|       1|       1|           1|1917.0300000000002|
|0670e0ca-bafc-4bd...|appliances.person...|       1|       0|           0|            100.81|
|2b9a9b2c-0837-425...|  apparel.shoes.keds|       1|       0|           0|            247.88|
|e9548b30-aed5-445...|  electronics.clocks|       1|       0|           0|           2174.97|
|30978e3b-4aa1-45c...|appliances.enviro...|       1|       1|           1|           1053.94|
+--------------------+--------------------+--------+--------+------------+------------------+
only showing top 5 rows


### 2. Identifying Revenue Leakage
We aggregate the 'lost' revenue (sessions with a view but no purchase) by category.

In [ ]:
# Calculate drop-offs and lost revenue
leakage_df = session_funnel.filter("category_code IS NOT NULL") \
    .groupBy("category_code") \
    .agg(sf.sum("has_view").alias("total_views"),
         sf.sum("has_cart").alias("total_carts"),
         sf.sum("has_purchase").alias("total_purchases"),
         sf.sum(sf.when(sf.col("has_purchase") == 0, sf.col("potential_revenue")).otherwise(0)).alias("lost_revenue")) \
    .withColumn("abandonment_rate", (1 - (sf.col("total_purchases") / sf.col("total_views"))) * 100) \
    .orderBy(sf.desc("lost_revenue"))

top_leaks = leakage_df.limit(5).toPandas()
display(top_leaks)

### 3. Deliverable: Merchandiser Dashboard
Visualizing the top 5 bleeding categories and the impact of a 5% recovery.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate 5% recovery projection
top_leaks['recovered_5pct'] = top_leaks['lost_revenue'] * 0.05

fig, ax1 = plt.subplots(figsize=(12, 6))

# Plot Lost Revenue
sns.barplot(x='category_code', y='lost_revenue', data=top_leaks, ax=ax1, palette='Reds_r')
ax1.set_title('Top 5 Categories by Lost Revenue (Friction Analysis)', fontsize=15)
ax1.set_ylabel('Total Lost Revenue ($)')
plt.xticks(rotation=45)

# Annotate with 5% recovery potential
for i, row in top_leaks.iterrows():
    ax1.text(i, row['lost_revenue'], f"5% Recovery:\n${row['recovered_5pct']:,.0f}",
             ha='center', va='bottom', fontweight='bold', color='green')

plt.tight_layout()
plt.show()

### Project A Summary: Identifying Zero-Result Friction

**Insights & Business Impact:**
- **Funnel Analysis:** By sessionizing hundreds of millions of rows, we identified specific categories where 'View-to-Purchase' conversion is lowest.
- **Revenue Leakage:** The dashboard above highlights the Top 5 categories responsible for the highest dollar-value abandonment.
- **Recovery Potential:** As shown in the projections, a modest **5% improvement** in search relevance or friction reduction in these categories can recover significant lost revenue (represented by the green annotations).

**Next Steps for Merchandisers:**
1. Review the 'abandonment_rate' for the identified categories.
2. Audit search results for these categories to identify 'Zero-Result' or irrelevant matches.
3. Implement synonym libraries or boost popular products to recover the projected revenue.